# AdS4 Indistinguishable Dual Solutions Test (v17)

Goal: construct two different geometries that fit the **same reduced probe set** almost equally well.

This notebook tests true non-identifiability by:
- using a reduced probe basis (EE + WL)
- training two solutions from different initializations / regularization bias
- comparing whether both solutions fit the same probes similarly
- showing that the recovered geometries can still differ

Interpretation:
- if A and B match EE + WL equally well but differ geometrically,
  the inverse problem is genuinely non-unique under that probe set


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Ground-truth AdS4-style toy background
r = torch.linspace(1.05, 3.5, 240, device=device).view(-1, 1)

def f_true(r):
    return r**2 + 1 - 1/r

def g_true(r):
    return 1/(f_true(r) + 1e-6)

def ee(f, g):
    return torch.log(1 + 0.6 * f)

def wl(f, g):
    return torch.sqrt(f + 1.8 * g + 1e-6)

ee_obs = ee(f_true(r), g_true(r)).detach()
wl_obs = wl(f_true(r), g_true(r)).detach()

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = MLP()
        self.g = MLP()
    def forward(self, r):
        return F.softplus(self.f(r)), F.softplus(self.g(r))

In [ ]:
def train(seed=0, bias_strength=0.0, epochs=1400, consistency_weight=0.15):
    torch.manual_seed(seed)
    np.random.seed(seed)

    m = Model().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)

    # bias term encourages a different geometry branch without changing probe targets
    radial_weight = (r - r.min()) / (r.max() - r.min())

    loss_hist = []
    for _ in range(epochs):
        opt.zero_grad()
        f, g = m(r)

        loss_ee = ((ee(f, g) - ee_obs)**2).mean()
        loss_wl = ((wl(f, g) - wl_obs)**2).mean()
        loss_cons = ((f * g - 1.0)**2).mean()

        # Branch-bias regularizer: pushes f slightly upward at large r and g slightly downward.
        # This keeps the same probe targets but nudges optimization toward an alternative geometry.
        bias = bias_strength * ((radial_weight * f).mean() - 0.25 * (radial_weight * g).mean())

        loss = loss_ee + loss_wl + consistency_weight * loss_cons + bias
        loss.backward()
        opt.step()
        loss_hist.append(loss.item())

    with torch.no_grad():
        f_pred, g_pred = m(r)
        ee_pred = ee(f_pred, g_pred)
        wl_pred = wl(f_pred, g_pred)
        metric_err = ((f_pred - f_true(r)).abs() + (g_pred - g_true(r)).abs()) / 2.0

    return {
        'f': f_pred.detach().cpu(),
        'g': g_pred.detach().cpu(),
        'ee_pred': ee_pred.detach().cpu(),
        'wl_pred': wl_pred.detach().cpu(),
        'loss_hist': loss_hist,
        'mean_err': metric_err.mean().item(),
        'max_err': metric_err.max().item(),
    }

In [ ]:
# Two solutions: same probe targets, different branch bias
run_A = train(seed=0, bias_strength=0.0)
run_B = train(seed=7, bias_strength=-0.08)

print('mean error A:', run_A['mean_err'])
print('mean error B:', run_B['mean_err'])

In [ ]:
# Compare probe residuals against the SAME reduced probe set
def mse(pred, target):
    return float(((pred - target)**2).mean().item())

probe_names = ['EE', 'WL']
targets = {
    'EE': ee_obs.cpu(),
    'WL': wl_obs.cpu(),
}
preds = {
    'A': {'EE': run_A['ee_pred'], 'WL': run_A['wl_pred']},
    'B': {'EE': run_B['ee_pred'], 'WL': run_B['wl_pred']},
}

resid = np.zeros((2, 2))
for i, sol in enumerate(['A', 'B']):
    for j, probe in enumerate(probe_names):
        resid[i, j] = mse(preds[sol][probe], targets[probe])

resid

In [ ]:
# Main figure: dual geometries + indistinguishable probe fit
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].plot(r.cpu(), f_true(r).cpu(), 'k', label='true')
axes[0,0].plot(r.cpu(), run_A['f'], label='A')
axes[0,0].plot(r.cpu(), run_B['f'], label='B')
axes[0,0].set_title('f(r): dual solutions')
axes[0,0].legend()

axes[0,1].plot(r.cpu(), g_true(r).cpu(), 'k', label='true')
axes[0,1].plot(r.cpu(), run_A['g'], label='A')
axes[0,1].plot(r.cpu(), run_B['g'], label='B')
axes[0,1].set_title('g(r): dual solutions')
axes[0,1].legend()

im = axes[1,0].imshow(resid, aspect='auto')
axes[1,0].set_xticks(range(len(probe_names)))
axes[1,0].set_xticklabels(probe_names)
axes[1,0].set_yticks([0,1])
axes[1,0].set_yticklabels(['A', 'B'])
axes[1,0].set_title('same-probe MSE')
fig.colorbar(im, ax=axes[1,0], fraction=0.046, pad=0.04)

x = np.arange(len(probe_names))
w = 0.35
axes[1,1].bar(x - w/2, resid[0], width=w, label='solution A')
axes[1,1].bar(x + w/2, resid[1], width=w, label='solution B')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(probe_names)
axes[1,1].set_title('reduced-probe fit comparison')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('ads4_phase_lock_v17_indistinguishable.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v17_indistinguishable.png')

In [ ]:
# Training trajectories
plt.figure(figsize=(8,4))
plt.plot(run_A['loss_hist'], label='solution A training')
plt.plot(run_B['loss_hist'], label='solution B training')
plt.title('training trajectories (same reduced probe set)')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.tight_layout()
plt.savefig('ads4_phase_lock_v17_losses.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v17_losses.png')